# Module 5: Train a Wordle Agent with GRPO

Fine-tune Qwen3-1.7B to play Wordle using GRPO (Group Relative Policy Optimization) via TRL and OpenEnv.

**Time:** ~90 min (training) · **Difficulty:** Advanced · **GPU:** A100 required (Colab Pro or similar)

Based on the [TRL OpenEnv Wordle example](https://github.com/huggingface/trl/blob/main/examples/notebooks/openenv_wordle_grpo.ipynb).

In [11]:
# Requires GPU (A100 recommended). Run on Colab Pro or similar.
# !pip install -Uq "trl>=0.17.0" openenv-core transformers datasets accelerate vllm trackio
# !git clone --depth=1 -q https://github.com/meta-pytorch/OpenEnv.git 2>/dev/null || true

import sys, os
repo = os.path.abspath('OpenEnv')
for p in [repo, os.path.join(repo, 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)
print("Setup complete!")

Setup complete!


In [8]:
# Optional Hugging Face auth (needed for gated models or pushing to Hub)
import os
from huggingface_hub import login, notebook_login

HF_TOKEN = os.getenv("HF_TOKEN")
HF_AUTH_OK = False

try:
    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=False)
        HF_AUTH_OK = True
        print("HF login successful via HF_TOKEN env var.")
    else:
        notebook_login()
        HF_AUTH_OK = True
        print("HF interactive login successful.")
except Exception as e:
    print(f"Skipping HF login in this environment: {e}")
    print("Set HF_TOKEN to enable non-interactive auth when sharing/running remotely.")

HF interactive login successful.


## 1. Initialize the Local Environment

Start (or connect to) a local TextArena Wordle server and create a persistent sync client.

> This notebook now prefers local server execution for development/testing.

In [12]:
import os
import socket
from urllib.parse import urlparse

from envs.textarena_env import TextArenaEnv

textarena_url = os.getenv("TEXTARENA_URL", "http://127.0.0.1:8001")

REMOTE_ENV_OK = False  # kept for downstream compatibility
sync_env = None
LOCAL_SERVER_STARTED = False


def _is_port_open(host: str, port: int, timeout_s: float = 0.5) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout_s)
        return sock.connect_ex((host, port)) == 0


parsed = urlparse(textarena_url)
host = parsed.hostname or "127.0.0.1"
port = parsed.port or 8001

if not _is_port_open(host, port):
    print("Local TextArena server is not running.")
    print("Start it first from /mnt/d/test/OpenEnv:")
    print(f"  uvicorn envs.textarena_env.server.app:app --host {host} --port {port}")
    raise RuntimeError("Local server required before running this cell.")

try:
    with TextArenaEnv(base_url=textarena_url, connect_timeout_s=20).sync() as _check:
        result = _check.reset()
        print(f"Connected to local env: {textarena_url}")
        print(f"Prompt preview: {str(result.observation.prompt)[:100]}...")

    env = TextArenaEnv(base_url=textarena_url, connect_timeout_s=20)
    sync_env = env.sync()
    sync_env.connect()
    REMOTE_ENV_OK = True
    print("Persistent local training connection established.")
except Exception as e:
    print(f"Local TextArena connection unavailable: {e}")
    print("Training/eval cells are guarded and will skip safely.")

Connected to local env: http://127.0.0.1:8001
Prompt preview: [GAME] You are Playing Wordle.
A secret 5-letter word has been chosen. You have 6 attempts to guess ...
Persistent local training connection established.


## 2. Init Model and Tokenizer

In [13]:
from transformers import AutoTokenizer
import torch

model_name = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

HAS_GPU = torch.cuda.is_available()
CAN_TRAIN = HAS_GPU and REMOTE_ENV_OK
print(f"Model: {model_name}")
print(f"GPU available: {HAS_GPU}")
print(f"Remote env connected: {REMOTE_ENV_OK}")
print(f"Training ready now: {CAN_TRAIN}")

Model: Qwen/Qwen3-1.7B
GPU available: True
Remote env connected: True
Training ready now: True


## 3. Define the System Prompt

In [14]:
system_prompt = """
You are an expert Wordle solver with deep knowledge of English vocabulary, letter frequency patterns, and optimal guessing strategies.

## GAME RULES

1. The target is a 5-letter English word
2. You have 6 attempts to guess the correct word
3. After each guess, you receive color-coded feedback:
   - GREEN: Letter is correct and in the correct position
   - YELLOW: Letter is in the word but in the wrong position
   - GRAY: Letter is not in the word at all
4. All guesses must be valid 5-letter English words
5. You cannot reuse a word you've already guessed

## RESPONSE FORMAT

Only respond with your next guess in square brackets, e.g., [crane].

## STRATEGIC APPROACH

Do not repeat the same guess twice.

### Opening Strategy
- Start with words rich in common vowels (A, E, I, O, U) and consonants (R, S, T, L, N)
- Optimal starters: CRANE, SLATE, STARE, AROSE, IRATE

### Mid-Game Strategy
- Use confirmed GREEN letters in their correct positions
- Place YELLOW letters in different positions than where they appeared
- Eliminate GRAY letters from consideration

## YOUR GOAL

Solve the Wordle in as few guesses as possible.
"""

## 4. Helper Functions

In [15]:
def make_user_prompt(prompt_text, messages):
    """Build a structured prompt from game state and message history."""
    history = format_history(messages)
    prompt_section = prompt_text.strip() if prompt_text.strip() else "Wordle-v0"
    history_section = history if history else "[PROMPT] Awaiting first feedback."
    return (
        f"Game prompt:\n{prompt_section}\n\n"
        f"Conversation so far:\n{history_section}\n\n"
        "Reply with your next guess enclosed in square brackets."
    )


def format_history(messages):
    """Format message history with category tags."""
    lines = []
    for message in messages:
        tag = message.category or "MESSAGE"
        content = message.content.strip()
        if content:
            lines.append(f"[{tag}] {content}")
    return "\n".join(lines)


def scale_repetition_score(previous_occurrences, max_occurrences):
    """Scale repetition penalty: 1.0 = novel guess, 0.0 = fully repeated."""
    if max_occurrences == 0:
        return 0.0
    return (max_occurrences - previous_occurrences) / max_occurrences


print("Helper functions defined.")

Helper functions defined.


## 5. Define the Rollout Function

The rollout function plays one full Wordle game per prompt. It's called by `GRPOTrainer` during training.

In [16]:
from collections import defaultdict
from envs.textarena_env.models import TextArenaAction
from envs.textarena_env.rewards import extract_feedback_counts, extract_guess, extract_wordle_feedback
from trl.experimental.openenv import generate_rollout_completions


def rollout_once(trainer, sync_env, tokenizer, dataset_prompt, system_prompt, max_turns):
    """Execute one full Wordle episode using an already-connected sync client."""
    result = sync_env.reset()
    observation = result.observation

    prompt_ids = []
    completion_ids = []
    logprobs = []
    green_scores = []
    yellow_scores = []
    repetition_scores = []
    correct_scores = []
    guess_counts = defaultdict(int)

    for _turn in range(max_turns):
        if result.done:
            break

        base_prompt = observation.prompt or dataset_prompt
        user_prompt = make_user_prompt(base_prompt, observation.messages)
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
            enable_thinking=False,
        )

        rollout_outputs = generate_rollout_completions(trainer, [prompt_text])[0]
        prompt_ids.extend(rollout_outputs['prompt_ids'])
        completion_ids.extend(rollout_outputs['completion_ids'])
        logprobs.extend(rollout_outputs['logprobs'])
        completion_text = rollout_outputs.get('text') or tokenizer.decode(
            rollout_outputs['completion_ids'], skip_special_tokens=True
        )

        guess = extract_guess(completion_text)
        result = sync_env.step(TextArenaAction(message=guess))
        observation = result.observation
        correct_score = float(result.reward or 0.0)
        feedback = extract_wordle_feedback(observation)

        previous_occurrences = guess_counts[guess]
        repetition_score = max(0.0, 1.0 - previous_occurrences)
        guess_counts[guess] += 1

        if not feedback:
            green_score, yellow_score = 0.0, 0.0
        else:
            green_count, yellow_count = extract_feedback_counts(feedback)
            green_score = green_count / 5.0
            yellow_score = yellow_count / 5.0

        repetition_scores.append(repetition_score)
        green_scores.append(green_score)
        yellow_scores.append(yellow_score)
        correct_scores.append(correct_score)

    return {
        'prompt_ids': prompt_ids,
        'completion_ids': completion_ids,
        'logprobs': logprobs,
        'correct_reward': correct_scores[-1] if correct_scores else 0.0,
        'green_reward': green_scores[-1] if green_scores else 0.0,
        'yellow_reward': yellow_scores[-1] if yellow_scores else 0.0,
        'repetition_reward': repetition_scores[-1] if repetition_scores else 0.0,
    }


def rollout_func(prompts, trainer=None):
    """Rollout function called by GRPOTrainer. Uses the module-level sync_env."""
    episode_prompt_ids = []
    episode_completion_ids = []
    episode_logprobs = []
    correctness_rewards = []
    green_rewards = []
    yellow_rewards = []
    repetition_rewards = []

    for prompt_text in prompts:
        episode = rollout_once(
            trainer=trainer,
            sync_env=sync_env,     # Persistent connection — no reconnect per episode
            tokenizer=tokenizer,
            dataset_prompt=prompt_text,
            system_prompt=system_prompt,
            max_turns=6,
        )
        episode_prompt_ids.append(episode['prompt_ids'])
        episode_completion_ids.append(episode['completion_ids'])
        episode_logprobs.append(episode['logprobs'])
        correctness_rewards.append(episode['correct_reward'])
        green_rewards.append(episode['green_reward'])
        yellow_rewards.append(episode['yellow_reward'])
        repetition_rewards.append(episode['repetition_reward'])

    return {
        'prompt_ids': episode_prompt_ids,
        'completion_ids': episode_completion_ids,
        'logprobs': episode_logprobs,
        'correct_reward': correctness_rewards,
        'green_reward': green_rewards,
        'yellow_reward': yellow_rewards,
        'repetition_reward': repetition_rewards,
    }


print('Rollout functions defined.')

/tmp/ipykernel_10086/1646347657.py:4: TRLExperimentalWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  from trl.experimental.openenv import generate_rollout_completions


Rollout functions defined.


## 6. Define Reward Functions

Four reward signals for richer gradient information.

In [17]:
def reward_correct(completions, **kwargs):
    rewards = kwargs.get("correct_reward")
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)

def reward_greens(completions, **kwargs):
    rewards = kwargs.get("green_reward")
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)

def reward_yellows(completions, **kwargs):
    rewards = kwargs.get("yellow_reward")
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)

def reward_repetition(completions, **kwargs):
    rewards = kwargs.get("repetition_reward")
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)

print("Reward functions: correct, greens, yellows, repetition")

Reward functions: correct, greens, yellows, repetition


## 7. Create Dataset

In [18]:
from datasets import Dataset

dataset_size = 1000
dataset = Dataset.from_dict({"prompt": ["Play Wordle like an expert."] * dataset_size})
print(f"Dataset: {len(dataset)} prompts")

Dataset: 1000 prompts


## 8. Configure GRPO Training

In [20]:
from inspect import signature
from trl import GRPOConfig

output_dir = "wordle-grpo-Qwen3-1.7B"

candidate_kwargs = {
    "num_train_epochs": 1,
    "learning_rate": 5e-6,
    "gradient_accumulation_steps": 64,
    "per_device_train_batch_size": 1,
    "warmup_steps": 20,
    "num_generations": 2,
    "max_completion_length": 8,
    "max_prompt_length": 1400,
    "use_vllm": HAS_GPU,
    "vllm_mode": "colocate",
    "vllm_gpu_memory_utilization": 0.1,
    "output_dir": output_dir,
    "report_to": "trackio",
    "trackio_space_id": output_dir,
    "logging_steps": 1,
    "save_steps": 10,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "push_to_hub": HF_AUTH_OK,
    "no_cuda": not HAS_GPU,
    "use_cpu": not HAS_GPU,
    "fp16": False,
    "bf16": False,
    "dataloader_num_workers": 0,
    "pin_memory": False,
    "remove_unused_columns": False,
    "max_steps": 1 if not HAS_GPU else -1,
    "save_strategy": "no" if not HAS_GPU else "steps",
}

supported = set(signature(GRPOConfig).parameters)
grpo_kwargs = {k: v for k, v in candidate_kwargs.items() if k in supported}
missing = sorted(set(candidate_kwargs) - set(grpo_kwargs))
if missing:
    print(f"Skipping unsupported GRPOConfig args for this trl version: {missing}")

grpo_config = GRPOConfig(**grpo_kwargs)

print(f"Output: {output_dir}")
print(f"vLLM enabled: {HAS_GPU}")
print(f"Push to Hub enabled: {HF_AUTH_OK}")

Skipping unsupported GRPOConfig args for this trl version: ['max_prompt_length', 'no_cuda', 'pin_memory']
Output: wordle-grpo-Qwen3-1.7B
vLLM enabled: True
Push to Hub enabled: True


## 9. Create Trainer and Train

In [21]:
from trl import GRPOTrainer

trainer = None
if CAN_TRAIN:
    trainer = GRPOTrainer(
        model=model_name,
        processing_class=tokenizer,
        reward_funcs=[
            reward_correct,
            reward_greens,
            reward_yellows,
            reward_repetition,
        ],
        train_dataset=dataset,
        args=grpo_config,
        rollout_func=rollout_func,
    )
    print("GRPOTrainer initialized.")
else:
    print("Skipping trainer initialization now (needs GPU + remote env).")

Fetching 2 files:   0%|          | 0/2 [02:17<?, ?it/s]


KeyboardInterrupt: 

In [22]:
# Check GPU before training
import torch
start_gpu_memory = 0.0
max_memory = 0.0

if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU: {gpu_stats.name} — {max_memory} GB total, {start_gpu_memory} GB reserved")
else:
    print("No CUDA GPU detected in this environment.")

GPU: NVIDIA GeForce RTX 3050 Laptop GPU — 4.0 GB total, 0.0 GB reserved


In [23]:
# Train (~90 minutes on A100)
trainer_stats = None
if trainer is not None and CAN_TRAIN:
    trainer_stats = trainer.train()
    print("Training completed.")
else:
    print("Skipping training (run this on a GPU machine with remote env access).")

Skipping training (run this on a GPU machine with remote env access).


In [24]:
# Memory stats after training (guarded for CPU / skipped training)
if trainer_stats is not None and torch.cuda.is_available() and max_memory > 0:
    used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    used_for_training = round(used_memory - start_gpu_memory, 3)

    print(f"Training time: {round(trainer_stats.metrics['train_runtime']/60, 1)} minutes")
    print(f"Peak memory: {used_memory} GB ({round(used_memory/max_memory*100, 1)}% of {max_memory} GB)")
    print(f"Memory for training: {used_for_training} GB")
else:
    print("Skipping post-training GPU stats (no completed GPU training run).")

Skipping post-training GPU stats (no completed GPU training run).


## 10. Save and Push

In [25]:
# Close environment connection and (optionally) save/push when training completed
if sync_env is not None:
    sync_env.close()
    print("Closed persistent env connection.")

if trainer is not None and trainer_stats is not None:
    trainer.save_model(output_dir)
    print(f"Model saved to {output_dir}.")

    if HF_AUTH_OK:
        trainer.push_to_hub()
        print("Model pushed to Hub.")
    else:
        print("Skipping Hub push (HF auth not available).")
else:
    print("Skipping save/push (no completed training run).")

if LOCAL_SERVER_STARTED and textarena_server_process is not None:
    textarena_server_process.terminate()
    textarena_server_process.wait(timeout=10)
    print("Stopped local TextArena server started by this notebook.")

Closed persistent env connection.
Skipping save/push (no completed training run).


## 11. Evaluate: Play Wordle with the Trained Model

In [26]:
import os
import torch
from transformers import AutoModelForCausalLM
from envs.textarena_env.models import TextArenaAction
from envs.textarena_env.rewards import extract_guess


def play_wordle(sync_env, model, tokenizer, max_turns=6):
    """Play one Wordle game and print each turn."""
    result = sync_env.reset()
    observation = result.observation
    print(f"Prompt: {observation.prompt[:100]}...")

    for turn in range(max_turns):
        if result.done:
            break

        user_prompt = make_user_prompt(observation.prompt, observation.messages)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
            enable_thinking=False,
        )

        model_inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)
        generated_ids = model.generate(**model_inputs, max_new_tokens=128)
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :]
        generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
        guess = extract_guess(generated_text)

        print(f"\nTurn {turn + 1}: {guess}")
        result = sync_env.step(TextArenaAction(message=guess))
        observation = result.observation
        for msg in observation.messages:
            print(f"  [{msg.category}] {msg.content}")

    print(f"\nResult: reward={result.reward}, done={result.done}")


if os.path.exists(output_dir):
    fine_tuned_model = AutoModelForCausalLM.from_pretrained(
        output_dir,
        torch_dtype="auto",
        device_map="auto" if torch.cuda.is_available() else None,
    )

    eval_env = TextArenaEnv(base_url=textarena_url)
    with eval_env.sync() as eval_sync:
        play_wordle(eval_sync, fine_tuned_model, tokenizer)
else:
    print(f"Skipping evaluation: model directory '{output_dir}' not found.")

Skipping evaluation: model directory 'wordle-grpo-Qwen3-1.7B' not found.


## Summary

What you did:
1. Connected to the TextArena Wordle environment via OpenEnv
2. Defined a system prompt, rollout function, and 4 reward signals
3. Trained Qwen3-1.7B with GRPO for ~90 minutes on an A100
4. Evaluated the trained model on live Wordle games

The key insight: **OpenEnv makes the environment a plug-in.** Swap Wordle for any other OpenEnv environment — your Module 4 word game, a coding environment, a math problem — and the training pipeline stays the same.

### What's next

- **Improve the model:** Longer training, larger models, better reward shaping
- **Build your own environment:** Use Module 4's pattern, plug it into this pipeline
- **Scale up:** See the [Scaling appendix](../README.md#bonus-scaling-openenv) for multi-container deployment
- **Explore the Hub:** Browse [openenv environments](https://huggingface.co/collections/openenv/environment-hub) for inspiration